In [ ]:
from pathlib import Path
import nest_asyncio
from rail.utils import catalog_utils
from rail_svc import db, local_sync
nest_asyncio.apply()

In [ ]:
try:
    await db.close_db()
except:
    pass
db.init_db()

#catalog_utils.load_yaml(catalog_utils.DEFAULT_CATAlOG_TAG_FILE)
local_sync.funcs.load_catalog_yaml(catalog_utils.DEFAULT_CATAlOG_TAG_FILE)

In [ ]:
model_dir_dp1_6band = Path('pz/projects/dp1_v4/data/gold_dp1_6band')
model_dir_dp1_4band = Path('pz/projects/dp1_v4/data/gold_dp1_4band')
estimates_dir_dp1_6band = Path('pz/projects/dp1_v4/data/gold_dp1_6band')
estimates_dir_dp1_4band = Path('pz/projects/dp1_v4/data/gold_dp1_4band')

In [ ]:
algos = {
    'knn':'rail.estimation.algos.k_nearneigh.KNearNeighEstimator',
    'fzboost':'rail.estimation.algos.flexzboost.FlexZBoostEstimator',
    'tpz':'rail.estimation.algos.tpz_lite.TPZliteEstimator',
    'cmnn':'rail.estimation.algos.cmnn.CMNNEstimator',
    'lephare':'rail.estimation.algos.lephare.LephareEstimator',
    'bpz':'rail.estimation.algos.bpz_lite.BPZliteEstimator',
    'gpz':'rail.estimation.algos.gpz.GPzEstimator',
    'dnf':'rail.estimation.algos.dnf.DNFEstimator',
}

In [ ]:
def safe_insert_catalog_tag(name):
    try:
        local_sync.catalog_tag.get_row_by_name(name)
    except:
        local_sync.catalog_tag.create_row(name=name)

In [ ]:
def safe_insert_dataset(name, path, catalog_tag_name, **kwargs):
    try:
        local_sync.dataset.get_row_by_name(name)
    except:
        local_sync.dataset.create_row(name=name, path=path, catalog_tag_name=catalog_tag_name, **kwargs)

In [ ]:
def safe_insert_algo(name, class_name):
    try:
        local_sync.algorithm.get_row_by_name(name)
    except:
        local_sync.algorithm.create_row(name=name, class_name=class_name)

In [ ]:
def safe_insert_model(name, path, algo_name, catalog_tag_name, **kwargs):
    try:
        local_sync.model.get_row_by_name(name)
    except:
        local_sync.model.create_row(name=name, path=path, algo_name=algo_name, catalog_tag_name=catalog_tag_name, **kwargs)

In [ ]:
def safe_insert_estimator(name, model_name, **kwargs):
    try:
        local_sync.estimator.get_row_by_name(name)
    except:
        local_sync.estimator.create_row(name=name, model_name=model_name, config=kwargs)

In [ ]:
def safe_insert_estimates(name, path, estimator_name, dataset_name, **kwargs):
    try:
        local_sync.estimates.get_row_by_name(name)
    except:
        local_sync.estimates.create_row(
            name=name, path=path, estimator_name=estimator_name, dataset_name=dataset_name, **kwargs
        )

In [ ]:
dataset_name = 'dp1_v4'
dataset_file = 'pz/data/test/dp1_matched_v4_test.hdf5'
catalog_tag_name = 'com_cam'

safe_insert_catalog_tag(catalog_tag_name)
safe_insert_dataset(dataset_name, dataset_file, catalog_tag_name, n_objects=-1, validate_file=False)

In [ ]:
for k, v in algos.items():
    algo = k
    catalog_tag_name = 'com_cam'
    model_name = f'dp1_6band_{algo}'
    estimator_name = f'dp1_6band_{algo}_base'
    estimates_name = f'dp1_6band_{algo}_base'
    model_file = str(model_dir_dp1_6band / f'model_inform_{algo}.pkl')
    qp_file = str(estimates_dir_dp1_6band / f'output_estimate_{algo}.hdf5')
    safe_insert_algo(name=algo, class_name=v)
    safe_insert_model(model_name, model_file, algo, catalog_tag_name, validate_file=False)
    safe_insert_estimator(estimator_name, model_name)
    if algo in ['dnf']:
        continue
    safe_insert_estimates(estimates_name, qp_file, estimator_name, dataset_name, n_objects=-1, validate_file=False)

In [ ]:
algos

In [ ]:
test = local_sync.funcs.estimate_pdf(1, 1, 4)

In [ ]:
test

In [ ]:
ret_vals = local_sync.funcs.get_dataset_and_estimates(1)

In [ ]:
ret_vals

In [ ]:
ret_vals = local_sync.funcs.get_data_and_estimates_data(1, 4)

In [ ]:
ret_vals

In [ ]:
local_sync.estimator.get_rows()